<a href="https://colab.research.google.com/github/thechiragbatra/customer-churn-mlops/blob/main/3_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ── CELL 1: Install dependencies ──────────────────────
!pip install fastapi uvicorn pydantic xgboost causalml mlflow nest-asyncio -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.

In [2]:
# ── CELL 2: Train + Save Both Models ──────────────────
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from causalml.inference.meta import BaseTClassifier
import pickle
from google.colab import files

# Load + clean data
uploaded = files.upload()
df = pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df.drop('customerID', axis=1, inplace=True)

# Encode
cat_cols = df.select_dtypes(include='object').columns.tolist()
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le

# Features
X = df.drop('Churn', axis=1)
y = df['Churn']
feature_names = X.columns.tolist()

# SMOTE + train churn model
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

churn_model = XGBClassifier(n_estimators=200, max_depth=6,
                             learning_rate=0.05, random_state=42,
                             eval_metric='logloss')
churn_model.fit(X_train_sm, y_train_sm)

# Train uplift model
np.random.seed(42)
df['treatment'] = np.random.binomial(1, 0.3, size=len(df))
uplift_model = BaseTClassifier(
    learner=XGBClassifier(n_estimators=100, max_depth=4,
                          random_state=42, eval_metric='logloss'))
uplift_model.fit(X=X.values, treatment=df['treatment'].values, y=y.values)

# Save both models
with open('churn_model.pkl', 'wb') as f:
    pickle.dump(churn_model, f)
with open('uplift_model.pkl', 'wb') as f:
    pickle.dump(uplift_model, f)
with open('feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

print("✅ Churn model saved!")
print("✅ Uplift model saved!")
print("✅ Feature names saved!")
print(f"\nFeatures: {feature_names}")

Saving WA_Fn-UseC_-Telco-Customer-Churn 2.csv to WA_Fn-UseC_-Telco-Customer-Churn 2.csv
✅ Churn model saved!
✅ Uplift model saved!
✅ Feature names saved!

Features: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


In [3]:
# ── CELL 3: FastAPI App ────────────────────────────────
import pickle
import numpy as np
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import uvicorn
import threading

nest_asyncio.apply()

# Load models
with open('churn_model.pkl', 'rb') as f:
    churn_model = pickle.load(f)
with open('uplift_model.pkl', 'rb') as f:
    uplift_model = pickle.load(f)
with open('feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

# Input schema
class Customer(BaseModel):
    gender: int
    SeniorCitizen: int
    Partner: int
    Dependents: int
    tenure: float
    PhoneService: int
    MultipleLines: int
    InternetService: int
    OnlineSecurity: int
    OnlineBackup: int
    DeviceProtection: int
    TechSupport: int
    StreamingTV: int
    StreamingMovies: int
    Contract: int
    PaperlessBilling: int
    PaymentMethod: int
    MonthlyCharges: float
    TotalCharges: float

app = FastAPI(title="Churn Prediction API", version="1.0")

@app.get("/")
def root():
    return {"message": "Churn Prediction API is live!", "status": "healthy"}

@app.post("/predict")
def predict(customer: Customer):
    # Prepare input
    input_data = np.array([[
        customer.gender, customer.SeniorCitizen, customer.Partner,
        customer.Dependents, customer.tenure, customer.PhoneService,
        customer.MultipleLines, customer.InternetService, customer.OnlineSecurity,
        customer.OnlineBackup, customer.DeviceProtection, customer.TechSupport,
        customer.StreamingTV, customer.StreamingMovies, customer.Contract,
        customer.PaperlessBilling, customer.PaymentMethod,
        customer.MonthlyCharges, customer.TotalCharges
    ]])

    # Churn probability
    churn_prob = float(churn_model.predict_proba(input_data)[0][1])

    # Uplift score
    ite = float(uplift_model.predict(X=input_data).flatten()[0])

    # Segment
    if churn_prob > 0.5 and ite < -0.05:
        segment = "Persuadable"
        recommendation = "🎯 Offer 20% discount immediately"
    elif churn_prob <= 0.5 and ite < -0.05:
        segment = "Sure Thing"
        recommendation = "✅ No action needed"
    elif churn_prob > 0.5 and ite >= -0.05:
        segment = "Lost Cause"
        recommendation = "❌ Accept churn — offer won't help"
    else:
        segment = "Sleeping Dog"
        recommendation = "⚠️ Do not contact — may trigger churn"

    # Top SHAP reasons
    import shap
    explainer = shap.TreeExplainer(churn_model)
    shap_vals = explainer.shap_values(input_data)[0]
    top_idx = np.argsort(np.abs(shap_vals))[::-1][:3]
    top_reasons = [feature_names[i] for i in top_idx]

    return {
        "churn_probability": round(churn_prob, 4),
        "uplift_score": round(ite, 4),
        "segment": segment,
        "recommendation": recommendation,
        "top_reasons": top_reasons
    }

# Run server in background thread
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

t = threading.Thread(target=run, daemon=True)
t.start()

import time
time.sleep(3)
print("✅ API is running on port 8000!")

INFO:     Started server process [11771]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ API is running on port 8000!


In [4]:
# ── CELL 4: Test the API ───────────────────────────────
import requests

# Sample customer — month-to-month contract, high charges, short tenure
test_customer = {
    "gender": 1,
    "SeniorCitizen": 0,
    "Partner": 0,
    "Dependents": 0,
    "tenure": 3,
    "PhoneService": 1,
    "MultipleLines": 0,
    "InternetService": 2,
    "OnlineSecurity": 0,
    "OnlineBackup": 0,
    "DeviceProtection": 0,
    "TechSupport": 0,
    "StreamingTV": 0,
    "StreamingMovies": 0,
    "Contract": 0,        # month-to-month
    "PaperlessBilling": 1,
    "PaymentMethod": 2,
    "MonthlyCharges": 85.5,
    "TotalCharges": 256.5
}

response = requests.post("http://localhost:8000/predict", json=test_customer)
result = response.json()

print("🎯 API Response:")
print(f"   Churn Probability : {result['churn_probability']}")
print(f"   Uplift Score      : {result['uplift_score']}")
print(f"   Segment           : {result['segment']}")
print(f"   Recommendation    : {result['recommendation']}")
print(f"   Top Reasons       : {result['top_reasons']}")

INFO:     127.0.0.1:56312 - "POST /predict HTTP/1.1" 200 OK
🎯 API Response:
   Churn Probability : 0.8685
   Uplift Score      : -0.0995
   Segment           : Persuadable
   Recommendation    : 🎯 Offer 20% discount immediately
   Top Reasons       : ['Contract', 'MonthlyCharges', 'InternetService']
